# **HW5 – QAOA and Applications**
_Time required: ~2–3 hours (for students with Qiskit optimization experience)_

**What you’ll practice**
- Formulating combinatorial problems (e.g., MaxCut) as QUBOs
- Building cost and mixer Hamiltonians
- Implementing QAOA circuits in Qiskit
- Variational optimization loops
- Evaluating approximation ratios
- Handling noise in QAOA
- Applications to graph problems in AI/DS

**What to turn in**
- This single notebook (`HW5_YourName.ipynb`) with **all cells run**, code and short written answers filled in where prompted.

**Rules & hints**
- Use **Qiskit** (version ~1.0 or later).
- Use AerSimulator for reproducibility; add noise where specified.
- If stuck, explain reasoning; partial credit for clear work.
- For graphs, use `networkx`; for optimization, use `scipy` or `qiskit.algorithms`.
- Use small graphs (n≤5) to avoid long runtimes.


In [ ]:
# --- Setup (run me first) ---
# Install Qiskit if needed (uncomment in Colab)
!pip install qiskit qiskit-aer qiskit-ibm-runtime qiskit-optimization networkx matplotlib scipy pylatexenc

# Import necessary modules
import matplotlib
import matplotlib.pyplot as plt
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph
import numpy as np
from scipy.optimize import minimize
from collections import defaultdict
from typing import Sequence

import pylatexenc
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_aer import AerSimulator

# Simulator backend
sim = AerSimulator()

# Function to run circuit and get counts
def get_counts(circ, shots=2000):
    tc = transpile(circ, sim)
    result = sim.run(tc, shots=shots).result()
    return result.get_counts()

def assert_close(A, B, tol=1e-8):
    if not np.allclose(A, B, atol=tol):
        raise AssertionError(f"Not close:\n{A}\nvs\n{B}")


## Part A (≈20 min)

**A1.** Create a K3 graph with weight 1 on each edge.  
**A2.** Create a list of pauli strings to encode the graph  
**A3.** Convert your pauli strings into a hamiltonian.


In [ ]:
# A1
n = 3

graph = rx.PyGraph()
graph.add_nodes_from(np.arange(0, n, 1))
# Your code here
edge_list = [
    (0, 1, 1.0),
    (0, 2, 1.0),
    (1, 2, 1.0),
]
graph.add_edges_from(edge_list)
draw_graph(graph, node_size=600, with_labels=True)

#A2
def build_max_cut_paulis(graph: rx.PyGraph) -> list[tuple[str, float]]:
    """Convert the graph to Pauli list.

    This function does the inverse of `build_max_cut_graph`
    """
    pauli_list = []
    for edge in list(graph.edge_list()):
        weight = graph.get_edge_data(edge[0], edge[1])
        # append the appropriate pauli string
        pauli_list.append((#ych, [edge[0], edge[1]], weight))


    return pauli_list

# A3 use SparsePauliOp.from_sparse_list to build the hamiltonian from you max_cut_paulis and n
max_cut_paulis = build_max_cut_paulis(graph)
# cost_hamiltonian = # Your code here
print("Cost Function Hamiltonian:", cost_hamiltonian)


# A4 use QAOAAnsatz to build the circuit with 2 reps
#circuit = QAOAAnsatz(cost_operator = # Your code here, ...)

circuit.measure_all()

circuit.draw("mpl")

## Part B — Run QAOA (≈25 min)

- **B1.** Load your runtime sevice and set the backend.
- **B2.** use the pass manager and fully optimize it.  
- **B3.** Set arbitrary parameters of pi and pi/2


In [ ]:
#B1 load your quantum account.  Set the backend as an AerSimulator()
your_api_key = ''
your_crn = ''

from qiskit_ibm_runtime import QiskitRuntimeService
QiskitRuntimeService.save_account(
  channel="do you know what to put here",
  token=your_api_key,
  instance=your_crn,
)

service = QiskitRuntimeService()
backend = # you code
print(backend)

In [ ]:
# Pass your circuit through the pass manager and fully optimize it
pm = # Your code here

candidate_circuit = pm.run(circuit)
candidate_circuit.draw("mpl", fold=False, idle_wires=False)

In [7]:
# B3 set gamma and beta parameters arbitrarily
initial_gamma = # ych
initial_beta = # ych
init_params = [initial_beta, initial_beta, initial_gamma, initial_gamma]

# C Define cost function and run using session
- C1. Define the primitive unitary block for the circuit
- C2. Enable dynamical decoupling and enable_gates for twirling
- C3. Assign result.x as a parameter and draw the cirucit

In [8]:
# C1
objective_func_vals = []
def cost_func_estimator(params, ansatz, hamiltonian, estimator):
    # transform the observable defined on virtual qubits to
    # an observable defined on all physical qubits
    isa_hamiltonian = hamiltonian.apply_layout(ansatz.layout)
    #C1 set pub as a variable with the ansatz, isa_hamiltonian, and params
    pub = # ych
    job = estimator.run([pub])

    results = job.result()[0]
    cost = results.data.evs

    objective_func_vals.append(cost)

    return cost

In [ ]:
# C2. Set shots to 1000, and apply dynamical decoupling and twirling for EM
with Session(backend=backend) as session:
    estimator = Estimator(mode=session)
    estimator.options.# ych

    # Set simple error suppression/mitigation options
    estimator.options.dynamical_decoupling.#ych
    estimator.options.dynamical_decoupling.sequence_type = "XY4"
    estimator.options.twirling.enable_gates#ych
    estimator.options.twirling.num_randomizations = "auto"

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit, cost_hamiltonian, estimator),
        method="COBYLA",
        tol=1e-2,
    )
    print(result)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

In [ ]:
# C3 assign `result.x` as the parameters and draw the cirucit
optimized_circuit = candidate_circuit.assign_parameters(# ych)
# ych

In [ ]:
# Initialize the sampler
sampler = Sampler(mode=backend)
sampler.options.default_shots = 10000

# Set simple error suppression/mitigation options
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XY4"
sampler.options.twirling.enable_gates = True
sampler.options.twirling.num_randomizations = "auto"

pub = (optimized_circuit,)
job = sampler.run([pub], shots=int(1e4))
counts_int = job.result()[0].data.meas.get_int_counts()
counts_bin = job.result()[0].data.meas.get_counts()
shots = sum(counts_int.values())
final_distribution_int = {key: val / shots for key, val in counts_int.items()}
final_distribution_bin = {key: val / shots for key, val in counts_bin.items()}
print(final_distribution_int)

In [ ]:
# auxiliary functions to sample most likely bitstring
def to_bitstring(integer, num_bits):
    result = np.binary_repr(integer, width=num_bits)
    return [int(digit) for digit in result]


keys = list(final_distribution_int.keys())
values = list(final_distribution_int.values())
most_likely = keys[np.argmax(np.abs(values))]
most_likely_bitstring = to_bitstring(most_likely, len(graph))
most_likely_bitstring.reverse()

print("Result bitstring:", most_likely_bitstring)

In [ ]:
matplotlib.rcParams.update({"font.size": 10})
final_bits = final_distribution_bin
values = np.abs(list(final_bits.values()))
top_4_values = sorted(values, reverse=True)[:4]
positions = []
for value in top_4_values:
    positions.append(np.where(values == value)[0])
fig = plt.figure(figsize=(11, 6))
ax = fig.add_subplot(1, 1, 1)
plt.xticks(rotation=45)
plt.title("Result Distribution")
plt.xlabel("Bitstrings (reversed)")
plt.ylabel("Probability")
ax.bar(list(final_bits.keys()), list(final_bits.values()), color="tab:grey")
for p in positions:
    ax.get_children()[int(p[0])].set_color("tab:purple")
plt.show()

In [ ]:
# auxiliary function to plot graphs
def plot_result(G, x):
    colors = ["tab:grey" if i == 0 else "tab:purple" for i in x]
    pos, _default_axes = rx.spring_layout(G), plt.axes(frameon=True)
    rx.visualization.mpl_draw(
        G, node_color=colors, node_size=100, alpha=0.8, pos=pos
    )


plot_result(graph, most_likely_bitstring)

# Part D - What have we done?
I need a full and explicit write up of what we have accomplished here?  What kind of problems can we solve with this?  How might we need to change the problem formulation?  How does this process scale in terms of width and depth?

In [ ]:
print("YOUR answer")

## Part E — Applications & Noise (≈20 min)
Run this cell.
Copy this cell 10 times with varying noise levels.
Write up a discussion about how noise affects these results and how you can use these functions to determine how resillient your algorithm is.


In [ ]:
# Import from Qiskit Aer noise module
# Example error probabilities
p_reset = 0.00
p_meas = 0.00
p_gate1 = 0.00

from qiskit_aer.noise import (
    NoiseModel,
    QuantumError,
    ReadoutError,
    depolarizing_error,
    pauli_error,
    thermal_relaxation_error,
)


# E1. Noisy QAOA
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.visualization import plot_histogram

noise_bit_flip = NoiseModel()

error_reset = pauli_error([("X", p_reset), ("I", 1 - p_reset)])
error_meas = pauli_error([("X", p_meas), ("I", 1 - p_meas)])
error_gate1 = pauli_error([("X", p_gate1), ("I", 1 - p_gate1)])
error_gate2 = error_gate1.tensor(error_gate1)

noise_bit_flip.add_all_qubit_quantum_error(error_reset, "reset")
noise_bit_flip.add_all_qubit_quantum_error(error_meas, "measure")
noise_bit_flip.add_all_qubit_quantum_error(error_gate1, ["u1", "u2", "u3"])
noise_bit_flip.add_all_qubit_quantum_error(error_gate2, ["cx"])

noise_model = AerSimulator(noise_model=noise_bit_flip)

# Transpile circuit for noisy basis gates
passmanager = generate_preset_pass_manager(
    optimization_level=3, backend=noise_model
)
circ_tnoise = passmanager.run(optimized_circuit)

# Run and get counts
result_bit_flip = noise_model.run(circ_tnoise).result()
counts_bit_flip = result_bit_flip.get_counts(0)

# Plot noisy output
plot_histogram(counts_bit_flip)



In [24]:
# your run # 1

In [25]:
# your run # 2

In [26]:
# your run # 3

In [27]:
# your run # 4

In [28]:
# your run # 5

In [29]:
# your run # 6

In [30]:
# your run # 7

In [31]:
# your run # 8

In [32]:
# your run # 9

In [33]:
# your run # 10


In [ ]:
print("Your answer")